# 04_quantum_channels_and_noise

Purpose:
- Understand quantum channels as physical processes
- Represent noise using Kraus operators
- Connect open-system evolution to density matrices

In [1]:
# Single-qubit amplitude damping channel via Kraus operators
# Apply to a density matrix and verify trace preservation.

import numpy as np
from qiskit.quantum_info import DensityMatrix, Statevector

# --- 1) Choose damping probability gamma in [0, 1] ---
gamma = 0.25

# --- 2) Define Kraus operators for amplitude damping ---
K0 = np.array([[1, 0],
               [0, np.sqrt(1 - gamma)]], dtype=complex)

K1 = np.array([[0, np.sqrt(gamma)],
               [0, 0]], dtype=complex)

# --- 3) Prepare an input state and its density matrix ---
# Example: starting from |1> so damping is visible
rho_in = DensityMatrix(Statevector([0, 1])).data

# --- 4) Apply the channel: rho_out = sum_k K_k rho_in K_k^\dagger ---
rho_out = K0 @ rho_in @ K0.conj().T + K1 @ rho_in @ K1.conj().T

# --- 5) Verify trace preservation and print results ---
print("rho_in:\n", rho_in)
print("\nrho_out:\n", rho_out)

print("\nTr(rho_in)  =", np.trace(rho_in))
print("Tr(rho_out) =", np.trace(rho_out))

# Optional: verify the Completely Positive Trace Preserving (CPTP) completeness relation sum_k K_k^\dagger K_k = I
completeness = K0.conj().T @ K0 + K1.conj().T @ K1
print("\nK0†K0 + K1†K1:\n", completeness)

rho_in:
 [[0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]

rho_out:
 [[0.25+0.j 0.  +0.j]
 [0.  +0.j 0.75+0.j]]

Tr(rho_in)  = (1+0j)
Tr(rho_out) = (0.9999999999999999+0j)

K0†K0 + K1†K1:
 [[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]


In [2]:
# Single-qubit dephasing (phase-flip / Pauli-Z) channel via Kraus operators
# Apply to |+> and show coherence decay with unchanged populations.

import numpy as np
from qiskit.quantum_info import DensityMatrix, Statevector

# --- 1) Dephasing strength p in [0, 1] ---
p = 0.4  # p=0 no dephasing, p=1 full dephasing

# --- 2) Kraus operators for dephasing ---
K0 = np.sqrt(1 - p) * np.eye(2, dtype=complex)
K1 = np.sqrt(p) * np.array([[1, 0],
                            [0, -1]], dtype=complex)  # Pauli-Z

# --- 3) Prepare |+> state and density matrix ---
psi_plus = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])
rho_in = DensityMatrix(psi_plus).data

# --- 4) Apply the channel: rho_out = sum_k K_k rho K_k^\dagger ---
rho_out = K0 @ rho_in @ K0.conj().T + K1 @ rho_in @ K1.conj().T

# --- 5) Inspect results ---
print("rho_in (|+>):\n", rho_in)
print("\nrho_out (after dephasing):\n", rho_out)

print("\nPopulations (diagonal):")
print("in :", np.diag(rho_in))
print("out:", np.diag(rho_out))

print("\nCoherence (off-diagonal magnitude):")
print("in :", abs(rho_in[0,1]))
print("out:", abs(rho_out[0,1]))

# --- 6) Verify trace preservation and completeness ---
print("\nTr(rho_in)  =", np.trace(rho_in))
print("Tr(rho_out) =", np.trace(rho_out))
print("\nK0†K0 + K1†K1:\n", K0.conj().T @ K0 + K1.conj().T @ K1)

rho_in (|+>):
 [[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]

rho_out (after dephasing):
 [[0.5+0.j 0.1+0.j]
 [0.1+0.j 0.5+0.j]]

Populations (diagonal):
in : [0.5+0.j 0.5+0.j]
out: [0.5+0.j 0.5+0.j]

Coherence (off-diagonal magnitude):
in : 0.4999999999999999
out: 0.09999999999999995

Tr(rho_in)  = (0.9999999999999998+0j)
Tr(rho_out) = (0.9999999999999998+0j)

K0†K0 + K1†K1:
 [[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]


Key takeaway:
- Quantum channels describe physical processes, not measurements
- Kraus operators encode environmental interaction
- Noise is just open-system dynamics

## Exercise 1: 

Let's apply the same dephasing channel to |0> and |+>. Then, let's compare 
1. The density matrices, 
2. The Z-basis probabilities (diagonal elements)
3. The X-basis expectation values (off-diagonal elements)

In [3]:
# Single-qubit dephasing (phase-flip / Pauli-Z) channel via Kraus operators
# Apply to |+> and show coherence decay with unchanged populations.

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix, Statevector

# --- 1) Dephasing strength p in [0, 1] ---
p = 0.4  # p=0 no dephasing, p=1 full dephasing

# --- 2) Kraus operators for dephasing ---
K0 = np.sqrt(1 - p) * np.eye(2, dtype=complex)
K1 = np.sqrt(p) * np.array([[1, 0],
                            [0, -1]], dtype=complex)  # Pauli-Z

# --- 3) Prepare |+> and |0> states, and their corresponding density matrices ---
psi_plus = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])
psi_0 = Statevector.from_instruction(QuantumCircuit(1)) # other way to create the state vector
rho_plus_in = DensityMatrix(psi_plus).data
rho_0_in = DensityMatrix(psi_0).data

# --- 4) Apply the channel: rho_out = sum_k K_k rho K_k^\dagger ---

def apply_kraus(rho, Ks):
    return sum(K @ rho @ K.conj().T for K in Ks)

#rho_plus_out = K0 @ rho_plus_in @ K0.conj().T + K1 @ rho_plus_in @ K1.conj().T
#rho_0_out = K0 @ rho_0_in @ K0.conj().T + K1 @ rho_0_in @ K1.conj().T
rho_plus_out = apply_kraus(rho_plus_in, [K0, K1])
rho_0_out    = apply_kraus(rho_0_in, [K0, K1])

# --- 5) Inspect results ---
print("rho_plus_in (|+>):\n", rho_plus_in)
print("\nrho_plus_out (after dephasing):\n", rho_plus_out)
print("\nrho_0_in (|0>):\n", rho_0_in)
print("\nrho_0_out (after dephasing):\n", rho_0_out)

print("\nPopulations (diagonal):")
print("rho_plus_in:", np.diag(rho_plus_in))
print("rho_0_in:", np.diag(rho_0_in))
print("rho_plus_out:", np.diag(rho_plus_out))
print("rho_0_out:", np.diag(rho_0_out))

print("\nCoherence (off-diagonal magnitude):")
print("rho_plus_in:", abs(rho_plus_in[0,1]))
print("rho_0_in:", abs(rho_0_in[0,1]))
print("rho_plus_out:", abs(rho_plus_out[0,1]))
print("rho_0_out:", abs(rho_0_out[0,1]))

# --- 6) Verify trace preservation and completeness ---
print("\nTr(rho_plus_in)  =", np.trace(rho_plus_in))
print("\nTr(rho_0_in)  =", np.trace(rho_0_in))
print("Tr(rho_plus_out) =", np.trace(rho_plus_out))
print("Tr(rho_0_out) =", np.trace(rho_0_out))
print("\nK0†K0 + K1†K1:\n", K0.conj().T @ K0 + K1.conj().T @ K1)

rho_plus_in (|+>):
 [[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]

rho_plus_out (after dephasing):
 [[0.5+0.j 0.1+0.j]
 [0.1+0.j 0.5+0.j]]

rho_0_in (|0>):
 [[1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]

rho_0_out (after dephasing):
 [[1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]

Populations (diagonal):
rho_plus_in: [0.5+0.j 0.5+0.j]
rho_0_in: [1.+0.j 0.+0.j]
rho_plus_out: [0.5+0.j 0.5+0.j]
rho_0_out: [1.+0.j 0.+0.j]

Coherence (off-diagonal magnitude):
rho_plus_in: 0.4999999999999999
rho_0_in: 0.0
rho_plus_out: 0.09999999999999995
rho_0_out: 0.0

Tr(rho_plus_in)  = (0.9999999999999998+0j)

Tr(rho_0_in)  = (1+0j)
Tr(rho_plus_out) = (0.9999999999999998+0j)
Tr(rho_0_out) = (1+0j)

K0†K0 + K1†K1:
 [[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]


## Exercise 2:

Let's apply the amplitude damping channel to |+>. Then, let's compute

1. The Z-basis probabilities via projectors
2. The X-basis expectation values

In [4]:
# Single-qubit amplitude damping channel via Kraus operators
# Apply to a density matrix and verify trace preservation.

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix, Statevector, Operator, Pauli
from qiskit.circuit.library import XGate

# --- 1) Choose damping probability gamma in [0, 1] ---
gamma = 0.25

# --- 2) Define Kraus operators for amplitude damping ---
K0 = np.array([[1, 0],
               [0, np.sqrt(1 - gamma)]], dtype=complex)

K1 = np.array([[0, np.sqrt(gamma)],
               [0, 0]], dtype=complex)

# --- 3) Prepare an input state |+> and its density matrix ---
#qc = QuantumCircuit(1)
#qc.h(0)
#Statevector.from_instruction(qc)
rho_in = DensityMatrix(Statevector([1/np.sqrt(2), 1/np.sqrt(2)])).data # more condensed

# --- 4) Apply the channel: rho_out = sum_k K_k rho_in K_k^\dagger ---
rho_out = K0 @ rho_in @ K0.conj().T + K1 @ rho_in @ K1.conj().T

# --- 5) Verify trace preservation and print results ---
print("rho_in:\n", rho_in)
print("\nrho_out:\n", rho_out)

print("\nTr(rho_in)  =", np.trace(rho_in))
print("Tr(rho_out) =", np.trace(rho_out))

# --- 5) Output Z-basis probabilities via projectors ---
print("\nZ-basis probabilities (diagonal elements):")
print("in:", np.real(np.diag(rho_in)))
print("out:", np.real(np.diag(rho_out)))

# --- 6) Output X-basis expectation values ---
#X = Pauli("X").to_matrix()
X = Operator(XGate()).data
expX_in  = np.real(np.trace(rho_in  @ X))
expX_out = np.real(np.trace(rho_out @ X))
print("\n <X> (trace formula):")
print("in:", expX_in)
print("out:", expX_out)

# Optional: verify the Completely Positive Trace Preserving (CPTP) completeness relation sum_k K_k^\dagger K_k = I
completeness = K0.conj().T @ K0 + K1.conj().T @ K1
print("\nK0†K0 + K1†K1:\n", completeness)

rho_in:
 [[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]

rho_out:
 [[0.625    +0.j 0.4330127+0.j]
 [0.4330127+0.j 0.375    +0.j]]

Tr(rho_in)  = (0.9999999999999998+0j)
Tr(rho_out) = (0.9999999999999998+0j)

Z-basis probabilities (diagonal elements):
in: [0.5 0.5]
out: [0.625 0.375]

 <X> (trace formula):
in: 0.9999999999999998
out: 0.8660254037844384

K0†K0 + K1†K1:
 [[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]


In [5]:
import numpy as np

def apply_kraus_channel(rho: np.ndarray, kraus_ops: list[np.ndarray]) -> np.ndarray:
    """
    Apply a quantum channel in Kraus form to a density matrix.

    Args:
        rho: Density matrix as a (d, d) complex numpy array.
        kraus_ops: List of Kraus operators K_k, each a (d, d) complex numpy array.

    Returns:
        rho_out: Output density matrix sum_k K_k rho K_k^\dagger.
    """
    rho = np.asarray(rho, dtype=complex)
    d1, d2 = rho.shape
    if d1 != d2:
        raise ValueError(f"rho must be square; got shape {rho.shape}")

    rho_out = np.zeros_like(rho, dtype=complex)
    for K in kraus_ops:
        K = np.asarray(K, dtype=complex)
        if K.shape != rho.shape:
            raise ValueError(f"Kraus operator shape {K.shape} does not match rho shape {rho.shape}")
        rho_out += K @ rho @ K.conj().T

    return rho_out

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/var/folders/s1/t5h9_47x4357q7sl4ykbgmrc0000gn/T/ipykernel_38468/1942137544.py:4: SyntaxWarning: invalid escape sequence '\d'
  """


In [6]:
import numpy as np
from qiskit.quantum_info import Operator

def pauli_expectation(rho: np.ndarray, pauli_op) -> float:
    """
    Compute the expectation value Tr(rho O) for a Pauli observable.

    Args:
        rho: Density matrix as a (d, d) complex numpy array.
        pauli_op: A Pauli observable. Can be:
                  - qiskit.quantum_info.Pauli (e.g., Pauli("X"))
                  - qiskit.circuit.library gate wrapped via Operator
                  - a (d, d) numpy array representing the observable

    Returns:
        Expectation value as a real float.
    """
    rho = np.asarray(rho, dtype=complex)

    if isinstance(pauli_op, Operator):
        O = pauli_op.data
    else:
        O = np.asarray(pauli_op, dtype=complex)

    if rho.shape != O.shape:
        raise ValueError(f"Shape mismatch: rho {rho.shape}, operator {O.shape}")

    return float(np.real(np.trace(rho @ O)))

In [7]:
from qiskit.circuit.library import XGate

X = Operator(XGate())
expX = pauli_expectation(rho_out, X)
print("<X> =", expX)

<X> = 0.8660254037844384
